<a href="https://colab.research.google.com/github/riyakumarif5-stack/IDRA-CAPSTONE-PROJECT/blob/main/IDRA_Capstone_Project_10_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Online Shopping Behaviour: Analysing Customer Activity and Predicting Purchase Intent

**IDRA Capstone Project — Project Topic 10**

Name: Riya Kumari | Institute: Manipal University Jaipur | Enrollment No:IDRA-2026-121357

---

This notebook contains the complete workflow: data loading, cleaning, EDA, statistical analysis,
feature engineering, model development, and evaluation.

**How to run in Colab:** Upload `P_10_Ecommerce.csv` using the cell in Section 1, then Runtime > Run all.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'
pd.set_option('display.max_columns', 50)

print('Libraries loaded.')

In [ ]:
# --- Load the dataset ---
# Option A (default): auto-download from GitHub raw URL - no upload needed.
#   Replace GITHUB_RAW_URL below with your own repo's raw link once uploaded.
# Option B: if the URL fails or you're testing locally, it falls back to
#   a manual upload prompt (Colab) or expects the CSV in the current folder.

import os

GITHUB_RAW_URL = "https://raw.githubusercontent.com/<your-username>/<your-repo>/main/P_10_Ecommerce.csv"
LOCAL_FILENAME = "P_10_Ecommerce.csv"

if not os.path.exists(LOCAL_FILENAME):
    try:
        import urllib.request
        urllib.request.urlretrieve(GITHUB_RAW_URL, LOCAL_FILENAME)
        print("Downloaded dataset from GitHub.")
    except Exception as e:
        print("Could not auto-download (", e, ") - falling back to manual upload.")
        try:
            from google.colab import files
            uploaded = files.upload()
        except ImportError:
            print("Not running in Colab - place P_10_Ecommerce.csv in this folder and re-run.")
else:
    print("Dataset already present locally.")


In [ ]:
df = pd.read_csv('P_10_Ecommerce.csv')
print('Shape:', df.shape)
df.head()

## 2. Dataset Understanding

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Target variable distribution
print(df['purchased'].value_counts())
print()
print(df['purchased'].value_counts(normalize=True).round(4) * 100)

### Category label mapping

The categorical columns are supplied as numeric codes with no data dictionary.
The labels below are **assumed** for readability in charts and are documented as an
assumption in the report (Section 5.3).

In [ ]:
device_map   = {0:'Desktop', 1:'Mobile', 2:'Tablet'}
user_map     = {0:'New', 1:'Returning'}
channel_map  = {0:'Direct', 1:'Organic Search', 2:'Paid Search',
                3:'Social Media', 4:'Email', 5:'Referral'}
category_map = {0:'Electronics', 1:'Fashion', 2:'Home & Kitchen', 3:'Beauty',
                4:'Sports', 5:'Books', 6:'Toys', 7:'Grocery'}
payment_map  = {0:'Credit Card', 1:'Debit Card', 2:'UPI',
                3:'Net Banking', 4:'Wallet', 5:'Cash on Delivery'}
season_map   = {0:'Winter', 1:'Spring', 2:'Summer', 3:'Autumn'}

df['device_type_lbl']       = df['device_type'].map(device_map)
df['user_type_lbl']         = df['user_type'].map(user_map)
df['marketing_channel_lbl'] = df['marketing_channel'].map(channel_map)
df['product_category_lbl']  = df['product_category'].map(category_map)
df['payment_method_lbl']    = df['payment_method'].map(payment_map)
df['visit_season_lbl']      = df['visit_season'].map(season_map)

df[['device_type_lbl','marketing_channel_lbl','product_category_lbl']].head()

## 3. Data Cleaning

In [ ]:
# 3.1 Missing values
print('Total missing values:', df.isnull().sum().sum())
print()
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# 3.2 Duplicate records
print('Full-row duplicates :', df.duplicated().sum())
print('Duplicate session_id:', df['session_id'].duplicated().sum())

In [ ]:
# 3.3 Data types - convert visit_date to datetime
df['visit_date'] = pd.to_datetime(df['visit_date'], format='%d-%m-%Y')
print(df['visit_date'].min(), 'to', df['visit_date'].max())
print(df['visit_date'].dtype)

In [ ]:
# 3.4 Outlier detection using the IQR method
for col in ['unit_price', 'time_on_site_sec', 'pages_viewed', 'quantity']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:20s} outliers: {n_out:5d}   actual range: {df[col].min()} - {df[col].max()}')

### 3.5 Data leakage check — the most important cleaning step

Several columns are only populated **after** a purchase decision has been made.
Using them as model inputs would leak the answer and produce misleadingly high accuracy.

In [ ]:
# Evidence of leakage: these fields behave completely differently by purchase outcome
leak_check = df.groupby('purchased')[['revenue', 'rating',
                                      'review_helpful_votes', 'cart_abandoned']].mean()
print(leak_check)
print()
print('revenue is exactly 0 for every non-purchase session:',
      (df.loc[df['purchased'] == 0, 'revenue'] == 0).all())

In [ ]:
LEAKAGE_COLS = ['rating', 'review_text', 'review_helpful_votes',
                'cart_abandoned', 'revenue', 'revenue_normalized']
print('Columns excluded from modelling due to leakage:')
for c in LEAKAGE_COLS:
    print(' -', c)

## 4. Exploratory Data Analysis

In [ ]:
# 4.1 Univariate - distribution of time on site
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.histplot(df['time_on_site_sec'], bins=30, kde=True, color='#2563eb', ax=ax)
ax.set_title('Distribution of Time on Site (seconds)', fontsize=13, fontweight='bold')
ax.set_xlabel('Time on Site (sec)'); ax.set_ylabel('Number of Sessions')
plt.show()

print(df['time_on_site_sec'].describe())

In [ ]:
# 4.2 Categorical - conversion rate by marketing channel
rate = df.groupby('marketing_channel_lbl')['purchased'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.barplot(x=rate.values, y=rate.index, hue=rate.index, palette='Blues_d', legend=False, ax=ax)
ax.set_title('Purchase Conversion Rate by Marketing Channel', fontsize=13, fontweight='bold')
ax.set_xlabel('Conversion Rate (%)'); ax.set_ylabel('')
for i, v in enumerate(rate.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.show()

print(rate.round(2))

In [ ]:
# 4.3 Bivariate - pages viewed vs purchase outcome
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.boxplot(x='purchased', y='pages_viewed', data=df, hue='purchased',
            palette=['#93c5fd', '#1d4ed8'], legend=False, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Not Purchased', 'Purchased'])
ax.set_title('Pages Viewed vs Purchase Outcome', fontsize=13, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Pages Viewed')
plt.show()

In [ ]:
# 4.4 Correlation heatmap
num_cols = ['unit_price','quantity','discount_percent','pages_viewed','time_on_site_sec',
            'added_to_cart','purchased','cart_abandoned','rating',
            'review_helpful_votes','visit_month']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap of Numerical Variables', fontsize=13, fontweight='bold')
plt.show()

# Note the high correlations for rating / review_helpful_votes / cart_abandoned --
# these are leakage artefacts, not genuine predictive signal.
print(df[num_cols].corr()['purchased'].sort_values(ascending=False).round(3))

In [ ]:
# 4.5 Purchase rate by device type
rate2 = df.groupby('device_type_lbl')['purchased'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(x=rate2.index, y=rate2.values, hue=rate2.index,
            palette='Blues_d', legend=False, ax=ax)
ax.set_title('Purchase Rate by Device Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Conversion Rate (%)'); ax.set_xlabel('')
for i, v in enumerate(rate2.values):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
plt.show()

## 5. Statistical Analysis

In [ ]:
# Descriptive statistics for key variables
key = ['unit_price','time_on_site_sec','pages_viewed','quantity','discount_percent']
desc = df[key].describe().T
desc['mode'] = df[key].mode().iloc[0]
desc[['mean','50%','mode','std','min','max']].round(2)

In [ ]:
# Chi-square test: is device type associated with purchasing?
ct = pd.crosstab(df['device_type_lbl'], df['purchased'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'Device type      chi2={chi2:.3f}, p={p:.4f}')

# Chi-square test: is marketing channel associated with purchasing?
ct2 = pd.crosstab(df['marketing_channel_lbl'], df['purchased'])
chi2b, pb, dofb, _ = stats.chi2_contingency(ct2)
print(f'Marketing channel chi2={chi2b:.3f}, p={pb:.4f}')

print()
print('Interpretation: p > 0.05 for both -> no statistically significant association.')

In [ ]:
# Independent t-test: do purchasers spend longer on site?
g1 = df.loc[df['purchased'] == 1, 'time_on_site_sec']
g0 = df.loc[df['purchased'] == 0, 'time_on_site_sec']
t, p_t = stats.ttest_ind(g1, g0, equal_var=False)
print(f'Time on site  -> purchasers: {g1.mean():.1f}s, non-purchasers: {g0.mean():.1f}s')
print(f'               t={t:.3f}, p={p_t:.6f}  (significant)')

print()

# Independent t-test: do purchasers view more pages?
g1b = df.loc[df['purchased'] == 1, 'pages_viewed']
g0b = df.loc[df['purchased'] == 0, 'pages_viewed']
t2, p2 = stats.ttest_ind(g1b, g0b, equal_var=False)
print(f'Pages viewed  -> purchasers: {g1b.mean():.2f}, non-purchasers: {g0b.mean():.2f}')
print(f'               t={t2:.3f}, p={p2:.4f}  (not significant)')

## 6. Feature Engineering

In [ ]:
# engagement_score combines browsing depth and duration into one measure
df['engagement_score'] = df['pages_viewed'] * df['time_on_site_sec'] / 1000

# is_weekend flags Saturday/Sunday sessions
df['is_weekend'] = df['visit_weekday'].isin([5, 6]).astype(int)

print(df.groupby('purchased')['engagement_score'].agg(['mean', 'median']).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.boxplot(x='purchased', y='engagement_score', data=df, hue='purchased',
            palette=['#93c5fd', '#1d4ed8'], legend=False, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Not Purchased', 'Purchased'])
ax.set_title('Engagement Score vs Purchase Outcome', fontsize=12, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Engagement Score')
plt.show()

## 7. Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop leakage columns, identifiers, and the label-only helper columns
ID_COLS = ['customer_id','session_id','product_id','visit_date','location',
           'device_type_lbl','user_type_lbl','marketing_channel_lbl',
           'product_category_lbl','payment_method_lbl','visit_season_lbl',
           'session_duration_bucket']

feature_cols = [c for c in df.columns
                if c not in LEAKAGE_COLS + ID_COLS + ['purchased']]

print(f'{len(feature_cols)} features used:')
print(feature_cols)

In [ ]:
X = df[feature_cols].copy()
y = df['purchased']

# One-hot encode the coded categorical variables
categorical = ['device_type','user_type','marketing_channel','product_category',
               'payment_method','visit_season','visit_weekday','is_weekend']
X = pd.get_dummies(X, columns=categorical, drop_first=True)

print('Shape after encoding:', X.shape)

In [ ]:
# Stratified 80/20 train-test split preserves the 22.5% purchase rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Purchase rate - train: %.4f, test: %.4f' % (y_train.mean(), y_test.mean()))

In [ ]:
# Scale numeric features (needed for Logistic Regression, not for Random Forest)
num_scale = ['unit_price','quantity','discount_percent','discount_amount',
             'pages_viewed','time_on_site_sec','visit_day','visit_month',
             'engagement_score']

scaler = StandardScaler()
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[num_scale] = scaler.fit_transform(X_train[num_scale])   # fit on train only
X_test_s[num_scale]  = scaler.transform(X_test[num_scale])

print('Scaling complete.')

## 8. Model Development

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve,
                             classification_report)

# class_weight='balanced' handles the 22.5% / 77.5% imbalance.
# Without it, the model just predicts "not purchased" for almost everything.

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_s, y_train)
pred_lr  = lr.predict(X_test_s)
proba_lr = lr.predict_proba(X_test_s)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
pred_rf  = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print('Both models trained.')

## 9. Model Evaluation

In [ ]:
def evaluate(name, model, X_tr, y_tr, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Train Acc': accuracy_score(y_tr, model.predict(X_tr)),
        'Test Acc':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'F1':        f1_score(y_true, y_pred),
        'ROC-AUC':   roc_auc_score(y_true, y_proba),
    }

results = pd.DataFrame([
    evaluate('Logistic Regression', lr, X_train_s, y_train, y_test, pred_lr, proba_lr),
    evaluate('Random Forest',       rf, X_train,   y_train, y_test, pred_rf, proba_rf),
]).set_index('Model').round(4)

results

In [ ]:
print('--- Logistic Regression ---')
print(classification_report(y_test, pred_lr))
print('--- Random Forest ---')
print(classification_report(y_test, pred_rf))

In [ ]:
# Confusion matrix for the final model (Random Forest)
cm = confusion_matrix(y_test, pred_rf)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Purchased', 'Purchased'],
            yticklabels=['Not Purchased', 'Purchased'])
ax.set_title('Confusion Matrix - Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.show()

In [ ]:
# Feature importance
imp = pd.Series(rf.feature_importances_, index=X_train.columns)\
        .sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.barplot(x=imp.values, y=imp.index, hue=imp.index, palette='Blues_d', legend=False, ax=ax)
ax.set_title('Top 10 Feature Importances - Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance'); ax.set_ylabel('')
plt.show()

print(imp.round(4))

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(6, 4.8))
for name, proba in [('Logistic Regression', proba_lr), ('Random Forest', proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, proba):.2f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontsize=12, fontweight='bold')
ax.legend()
plt.show()

## 10. Findings Summary

1. **Cart addition is the strongest behavioural predictor** — `added_to_cart` accounts for
   roughly 67% of total feature importance in the Random Forest.
2. **Device type and marketing channel do not significantly affect conversion**
   (chi-square p = 0.63 and p = 0.34 respectively).
3. **Longer time on site is weakly but significantly associated with purchasing**
   (929.7s vs 895.6s, t-test p < 0.001).
4. **Six fields were post-purchase artefacts** and had to be excluded to prevent data leakage.
5. **Final model:** Random Forest — 61% accuracy, 91% recall on purchasers, ROC-AUC 0.76.
   High recall is favoured because missing a genuine buyer costs more than an unnecessary offer.

### Recommendations
- Trigger retention interventions (exit-intent offers, cart reminders) on real-time cart
  activity and engagement score.
- Base marketing channel budget on traffic volume and cost, not assumed conversion differences.
- Always run a leakage audit as the first step in any purchase-prediction analysis.